# Tanzania Tourism Expenditure Classifier
**Stacking Pipeline**: Multi-Seed Stratified K-Fold Gradient Boosting Ensemble (LightGBM + CatBoost + XGBoost) with SLSQP Simplex Optimization.

## 1. Setup & Environment Dependencies

In [ ]:
%pip install catboost lightgbm xgboost scikit-learn pandas numpy scipy -q

import gc
import re
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier

In [ ]:
files = {
    'train': {
        'local': '../data/raw/Train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Train.csv'
    },
    'test': {
        'local': '../data/raw/Test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Test.csv'
    },
    'sample_sub': {
        'local': '../data/raw/SampleSubmission.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/SampleSubmission.csv'
    }
}

data = {}
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} locally.")
    except (FileNotFoundError, OSError):
        data[name] = pd.read_csv(paths['remote'])
        print(f"Loaded {name} from GitHub.")

train = data['train']
test = data['test']
sample_sub = data['sample_sub']

In [ ]:
SEED = 42
N_SEEDS = 3
N_FOLDS = 5
EPS = 1e-15

ID_COL = 'Tour_ID'
TARGET_COL = 'cost_category'
TARGET_CLASSES = ['High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']
NUM_CLASSES = len(TARGET_CLASSES)

CLASS_TO_IDX = {cls_name: i for i, cls_name in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {i: cls_name for i, cls_name in enumerate(TARGET_CLASSES)}
train['target'] = train[TARGET_COL].map(CLASS_TO_IDX)

## 2. Feature Engineering Pipeline

In [ ]:
def preprocess_dataset(df):
    df = df.copy()
    
    if 'main_activity' in df.columns:
        df['main_activity'] = df['main_activity'].replace({'Widlife Tourism': 'Wildlife Tourism'})
        
    df['travel_with'] = df['travel_with'].fillna('Alone')
    df['total_female'] = df['total_female'].fillna(0)
    df['total_male'] = df['total_male'].fillna(0)
    df['most_impressing'] = df.get('most_impressing', pd.Series(index=df.index)).fillna('No Answer')
    
    # Demographics & Group Sizing
    df['total_people'] = df['total_female'] + df['total_male']
    df['total_people_safe'] = df['total_people'].replace(0, 1)
    df['female_ratio'] = df['total_female'] / df['total_people_safe']
    
    # Duration Metrics
    df['total_nights'] = df['night_mainland'] + df['night_zanzibar']
    df['total_nights_safe'] = df['total_nights'].replace(0, 1)
    df['mainland_ratio'] = df['night_mainland'] / df['total_nights_safe']
    df['zanzibar_ratio'] = df['night_zanzibar'] / df['total_nights_safe']
    df['nights_per_person'] = df['total_nights'] / df['total_people_safe']
    
    # Tour Package Components
    package_cols = [
        'package_transport_int', 'package_accomodation', 'package_food',
        'package_transport_tz', 'package_sightseeing', 'package_guided_tour',
        'package_insurance'
    ]
    df['package_count'] = (df[package_cols] == 'Yes').sum(axis=1)
    df['package_depth'] = df['package_count'] / len(package_cols)
    df['is_full_package'] = (df['package_count'] == len(package_cols)).astype(int)
    df['has_no_package'] = (df['package_count'] == 0).astype(int)
    
    return df

train_df = preprocess_dataset(train)
test_df = preprocess_dataset(test)

CAT_FEATURES = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity',
    'info_source', 'tour_arrangement', 'package_transport_int',
    'package_accomodation', 'package_food', 'package_transport_tz',
    'package_sightseeing', 'package_guided_tour',
    'package_insurance', 'first_trip_tz', 'most_impressing'
]

# Joint Frequency Encoding
for col in ['country', 'purpose', 'main_activity']:
    freq_map = pd.concat([train_df[col], test_df[col]]).value_counts()
    train_df[f'{col}_freq'] = train_df[col].map(freq_map)
    test_df[f'{col}_freq'] = test_df[col].map(freq_map)

feature_cols = [c for c in train_df.columns if c not in [ID_COL, TARGET_COL, 'target']]

# Format 1: CatBoost Raw String Features
X_tr_cb = train_df[feature_cols].copy()
X_te_cb = test_df[feature_cols].copy()
for c in CAT_FEATURES:
    X_tr_cb[c] = X_tr_cb[c].fillna('Missing').astype(str)
    X_te_cb[c] = X_te_cb[c].fillna('Missing').astype(str)

# Format 2: LightGBM & XGBoost One-Hot Encoding
full_df = pd.concat([train_df[feature_cols], test_df[feature_cols]], axis=0).reset_index(drop=True)
full_df_encoded = pd.get_dummies(full_df, columns=CAT_FEATURES, drop_first=False)
full_df_encoded.columns = [re.sub(r'[^a-zA-Z0-9_]+', '_', str(c)).strip('_') for c in full_df_encoded.columns]

X_train = full_df_encoded.iloc[:len(train_df)].copy()
X_test = full_df_encoded.iloc[len(train_df):].copy()
y = train_df['target'].values

## 3. Model Hyperparameter Factories

In [ ]:
def build_lgb(seed):
    return lgb.LGBMClassifier(
        objective="multiclass",
        num_class=NUM_CLASSES,
        metric="multi_logloss",
        learning_rate=0.03,
        num_leaves=48,
        min_child_samples=25,
        subsample=0.85,
        subsample_freq=5,
        colsample_bytree=0.75,
        reg_alpha=0.3,
        reg_lambda=0.6,
        n_estimators=3000,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )

def build_cat(seed):
    return CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=3000,
        learning_rate=0.04,
        depth=6,
        l2_leaf_reg=4.0,
        random_strength=1.0,
        subsample=0.85,
        random_seed=seed,
        early_stopping_rounds=200,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1
    )

def build_xgb(seed):
    return XGBClassifier(
        objective="multi:softprob",
        num_class=NUM_CLASSES,
        eval_metric="mlogloss",
        tree_method="hist",
        learning_rate=0.04,
        max_depth=6,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_alpha=0.3,
        reg_lambda=0.8,
        n_estimators=3000,
        early_stopping_rounds=150,
        random_state=seed,
        n_jobs=-1,
        verbosity=0
    )

## 4. Multi-Seed Stratified K-Fold Cross-Validation

In [ ]:
oof_lgb = np.zeros((len(X_train), NUM_CLASSES))
oof_cat = np.zeros((len(X_train), NUM_CLASSES))
oof_xgb = np.zeros((len(X_train), NUM_CLASSES))

test_lgb = np.zeros((len(X_test), NUM_CLASSES))
test_cat = np.zeros((len(X_test), NUM_CLASSES))
test_xgb = np.zeros((len(X_test), NUM_CLASSES))

print(f"--- Executing Multi-Seed Cross-Validation ({N_SEEDS} Seeds x {N_FOLDS} Folds) ---")

for s in range(N_SEEDS):
    seed_val = SEED + s
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed_val)
    
    seed_oof_lgb = np.zeros_like(oof_lgb)
    seed_oof_cat = np.zeros_like(oof_cat)
    seed_oof_xgb = np.zeros_like(oof_xgb)
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y)):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        
        # LightGBM
        model_lgb = build_lgb(seed_val)
        model_lgb.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(150, verbose=False)]
        )
        seed_oof_lgb[va_idx] = model_lgb.predict_proba(X_va)
        test_lgb += model_lgb.predict_proba(X_test) / (N_FOLDS * N_SEEDS)
        
        # CatBoost
        model_cat = build_cat(seed_val)
        tr_pool = Pool(X_tr_cb.iloc[tr_idx], y_tr, cat_features=CAT_FEATURES)
        va_pool = Pool(X_tr_cb.iloc[va_idx], y_va, cat_features=CAT_FEATURES)
        model_cat.fit(tr_pool, eval_set=va_pool, use_best_model=True)
        seed_oof_cat[va_idx] = model_cat.predict_proba(va_pool)
        test_cat += model_cat.predict_proba(Pool(X_te_cb, cat_features=CAT_FEATURES)) / (N_FOLDS * N_SEEDS)
        
        # XGBoost
        model_xgb = build_xgb(seed_val)
        model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        seed_oof_xgb[va_idx] = model_xgb.predict_proba(X_va)
        test_xgb += model_xgb.predict_proba(X_test) / (N_FOLDS * N_SEEDS)
        
        print(f"[Seed {s+1}/{N_SEEDS} | Fold {fold+1}/{N_FOLDS}] "
              f"LGB: {log_loss(y_va, seed_oof_lgb[va_idx]):.5f} | "
              f"CAT: {log_loss(y_va, seed_oof_cat[va_idx]):.5f} | "
              f"XGB: {log_loss(y_va, seed_oof_xgb[va_idx]):.5f}")
        
    oof_lgb += seed_oof_lgb / N_SEEDS
    oof_cat += seed_oof_cat / N_SEEDS
    oof_xgb += seed_oof_xgb / N_SEEDS
    gc.collect()

## 5. Simplex Blending Weight Optimization

In [ ]:
oof_lgb_cal = np.clip(oof_lgb, EPS, 1 - EPS)
oof_cat_cal = np.clip(oof_cat, EPS, 1 - EPS)
oof_xgb_cal = np.clip(oof_xgb, EPS, 1 - EPS)

print("\n--- Out-Of-Fold (OOF) Baseline Log Loss Scores ---")
print(f" LightGBM OOF Log Loss : {log_loss(y, oof_lgb_cal):.5f}")
print(f" CatBoost OOF Log Loss : {log_loss(y, oof_cat_cal):.5f}")
print(f" XGBoost  OOF Log Loss : {log_loss(y, oof_xgb_cal):.5f}")

def objective(weights):
    blend = weights[0] * oof_lgb_cal + weights[1] * oof_cat_cal + weights[2] * oof_xgb_cal
    blend = np.clip(blend, EPS, 1 - EPS)
    blend /= blend.sum(axis=1, keepdims=True)
    return log_loss(y, blend)

# Constrained SLSQP Optimization (Sum == 1.0, Weights >= 0.0)
init_weights = [1/3, 1/3, 1/3]
bounds = [(0, 1), (0, 1), (0, 1)]
constraints = {'type': 'eq', 'fun': lambda w: 1.0 - sum(w)}

res = minimize(objective, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
w_lgb, w_cat, w_xgb = res.x

print("\n--- Optimal Constrained Ensemble Weights ---")
print(f" LightGBM Weight : {w_lgb:.4f}")
print(f" CatBoost Weight : {w_cat:.4f}")
print(f" XGBoost  Weight : {w_xgb:.4f}")

## 6. Final Blending & Submission File Generation

In [ ]:
oof_blend = w_lgb * oof_lgb_cal + w_cat * oof_cat_cal + w_xgb * oof_xgb_cal
oof_blend = np.clip(oof_blend, EPS, 1 - EPS)
oof_blend /= oof_blend.sum(axis=1, keepdims=True)

test_blend = w_lgb * test_lgb + w_cat * test_cat + w_xgb * test_xgb
test_blend = np.clip(test_blend, EPS, 1 - EPS)
test_blend /= test_blend.sum(axis=1, keepdims=True)

final_score = log_loss(y, oof_blend)
print(f"\nFINAL OPTIMIZED ENSEMBLE OOF LOG LOSS: {final_score:.5f}")

submission = pd.DataFrame(test_blend, columns=[IDX_TO_CLASS[i] for i in range(NUM_CLASSES)])
submission.insert(0, ID_COL, test_df[ID_COL])
submission = sample_sub[[ID_COL]].merge(submission, on=ID_COL, how='left')
submission.to_csv('final_ensemble_submission.csv', index=False)

print("Submission saved to 'final_ensemble_submission.csv'.")